**Objective**

Extract standardized image features from the sampled DMS dataset.

Input: data/samples/
Output: data/processed/dms_features.csv

In [1]:
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

In [2]:
PROJECT_DIR = Path(
    "/workspaces/DBA-ai-trust-score-framework-Interim"
)

SAMPLES_DIR = PROJECT_DIR / "data" / "samples"

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
metadata = pd.read_csv(
    PROCESSED_DIR / "dms_cleaned.csv"
)

print("Metadata shape:", metadata.shape)

print("\nColumns:")
print(metadata.columns.tolist())

metadata.head()

Metadata shape: (1400, 4)

Columns:
['file_name', 'label', 'dataset', 'original_path']


,file_name,label,dataset,original_path
0,gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
1,gA_4_s2_ir_face_mp4-372_jpg.rf.448fec2e90c845f...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
2,gA_5_s2_ir_face_mp4-130_jpg.rf.1081aec5f8fd8da...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
3,gB_9_s2_2019-03-07T16-21-20-01-00_ir_face_mp4-...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...
4,gA_5_s2_ir_face_mp4-246_jpg.rf.f7bcf481f361e68...,DangerousDriving,Thesis_dataset2,../data/raw/Thesis_dataset2/DangerousDriving/g...


In [4]:
LABEL_FOLDER_MAP = {

    "ClosedEye": "closed",

    "OpenEye": "Openeye",

    "DangerousDriving": "DangerousDriving",

    "Distracted": "Distracted",

    "Drinking": "Drinking",

    "SafeDriving": "SafeDriving",

    "Yawn": "Yawn"

}

In [5]:
def extract_features(image_path):

    # -----------------------------------------
    # Read image
    # -----------------------------------------

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    # -----------------------------------------
    # Convert to grayscale
    # -----------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # -----------------------------------------
    # Image dimensions
    # -----------------------------------------

    height, width = gray.shape

    image_area = width * height

    aspect_ratio = width / height

    # -----------------------------------------
    # File properties
    # -----------------------------------------

    size_mb = (
        image_path.stat().st_size
        / (1024 * 1024)
    )

    size_per_pixel = (
        size_mb / image_area
    )

    # -----------------------------------------
    # Image quality
    # -----------------------------------------

    brightness = np.mean(gray)

    contrast = np.std(gray)

    blur_score = cv2.Laplacian(
        gray,
        cv2.CV_64F
    ).var()

    # -----------------------------------------
    # Entropy
    # -----------------------------------------

    histogram = cv2.calcHist(
        [gray],
        [0],
        None,
        [256],
        [0, 256]
    )

    histogram = (
        histogram / histogram.sum()
    )

    entropy = -np.sum(
        histogram *
        np.log2(histogram + 1e-10)
    )

    # -----------------------------------------
    # Edge features
    # -----------------------------------------

    edges = cv2.Canny(
        gray,
        100,
        200
    )

    edge_density = np.mean(
        edges > 0
    )

    edge_strength = np.mean(edges)

    # -----------------------------------------
    # Dynamic range
    # -----------------------------------------

    dynamic_range = (
        gray.max() - gray.min()
    )

    # -----------------------------------------
    # Return features
    # -----------------------------------------

    return {

        "width": width,

        "height": height,

        "image_area": image_area,

        "aspect_ratio": aspect_ratio,

        "size_mb": size_mb,

        "size_per_pixel": size_per_pixel,

        "brightness": brightness,

        "contrast": contrast,

        "blur_score": blur_score,

        "entropy": entropy,

        "edge_density": edge_density,

        "edge_strength": edge_strength,

        "dynamic_range": dynamic_range,

        "is_square": int(
            abs(width - height) <= 10
        )

    }

In [6]:
test_row = metadata.iloc[0]

print("Label:", test_row["label"])

print("File:", test_row["file_name"])

folder = LABEL_FOLDER_MAP[
    test_row["label"]
]

test_path = (
    SAMPLES_DIR
    / folder
    / test_row["file_name"]
)

print("\nPath:")
print(test_path)

print("\nExists:")
print(test_path.exists())

Label: DangerousDriving
File: gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-584_jpg.rf.3eaade6cb9c646602f5b0899f5da6d6b.jpg

Path:
/workspaces/DBA-ai-trust-score-framework-Interim/data/samples/DangerousDriving/gB_7_s2_2019-03-11T14-12-25-01-00_ir_face_mp4-584_jpg.rf.3eaade6cb9c646602f5b0899f5da6d6b.jpg

Exists:
True


In [7]:
test_features = extract_features(
    test_path
)

print(test_features)

{'width': 561, 'height': 640, 'image_area': 359040, 'aspect_ratio': 0.8765625, 'size_mb': 0.024924278259277344, 'size_per_pixel': 6.941922420698903e-08, 'brightness': np.float64(54.612953988413544), 'contrast': np.float64(35.9717968426642), 'blur_score': np.float64(16.30326827372498), 'entropy': np.float32(6.8541374), 'edge_density': np.float64(0.007909982174688057), 'edge_strength': np.float64(2.0170454545454546), 'dynamic_range': np.uint8(249), 'is_square': 0}


In [8]:
feature_rows = []

missing_paths = 0
failed_images = 0

for _, row in tqdm(
    metadata.iterrows(),
    total=len(metadata)
):

    folder = LABEL_FOLDER_MAP[
        row["label"]
    ]

    image_path = (
        SAMPLES_DIR
        / folder
        / row["file_name"]
    )

    # Check image exists
    if not image_path.exists():

        missing_paths += 1

        continue

    # Extract image features
    features = extract_features(
        image_path
    )

    # Skip unreadable images
    if features is None:

        failed_images += 1

        continue

    # Combine metadata + image features
    feature_rows.append({

        **row.to_dict(),

        **features

    })

print("\nImages processed:", len(feature_rows))

print("Missing paths:", missing_paths)

print("Unreadable images:", failed_images)

  5%|▌         | 71/1400 [00:00<00:16, 79.97it/s]

100%|██████████| 1400/1400 [00:13<00:00, 105.48it/s]


Images processed: 1400
Missing paths: 0
Unreadable images: 0


In [9]:
feature_dataset = pd.DataFrame(
    feature_rows
)

print(
    "Feature dataset shape:",
    feature_dataset.shape
)

Feature dataset shape: (1400, 18)


cleaned metadata
       +
image-derived features
       =
feature_dataset

In [10]:
print(
    feature_dataset.columns.tolist()
)

['file_name', 'label', 'dataset', 'original_path', 'width', 'height', 'image_area', 'aspect_ratio', 'size_mb', 'size_per_pixel', 'brightness', 'contrast', 'blur_score', 'entropy', 'edge_density', 'edge_strength', 'dynamic_range', 'is_square']


In [11]:
print(
    feature_dataset.isnull().sum()
)

file_name         0
label             0
dataset           0
original_path     0
width             0
height            0
image_area        0
aspect_ratio      0
size_mb           0
size_per_pixel    0
brightness        0
contrast          0
blur_score        0
entropy           0
edge_density      0
edge_strength     0
dynamic_range     0
is_square         0
dtype: int64


In [12]:
feature_dataset.describe(
    include="all"
).T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
file_name,1400,1398,_59.png,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
label,1400,7,DangerousDriving,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dataset,1400,1,Thesis_dataset2,1400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_path,1400,1400,../data/raw/Thesis_dataset2/DangerousDriving/g...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
width,1400.0,NaN,NaN,NaN,454.19,234.502322,90.0,90.0,599.0,640.0,683.0
height,1400.0,NaN,NaN,NaN,458.04,235.939074,90.0,90.0,640.0,640.0,640.0
image_area,1400.0,NaN,NaN,NaN,258480.050714,161381.811029,8100.0,8100.0,307200.0,409600.0,409600.0
aspect_ratio,1400.0,NaN,NaN,NaN,1.008735,0.180968,0.6,0.989062,1.0,1.0,1.778646
size_mb,1400.0,NaN,NaN,NaN,0.030399,0.026165,0.002547,0.012955,0.023805,0.042636,0.341972
size_per_pixel,1400.0,NaN,NaN,NaN,0.0,0.000001,0.0,0.0,0.0,0.0,0.000002


In [13]:
FEATURE_FILE = (
    PROCESSED_DIR /
    "dms_features.csv"
)

feature_dataset.to_csv(
    FEATURE_FILE,
    index=False
)

print(
    "✅ dms_features.csv saved"
)

print(
    "Location:",
    FEATURE_FILE
)

✅ dms_features.csv saved
Location: /workspaces/DBA-ai-trust-score-framework-Interim/data/processed/dms_features.csv


In [14]:
print("=" * 60)
print("FINAL FEATURE DATASET VALIDATION")
print("=" * 60)

print(
    "Rows:",
    feature_dataset.shape[0]
)

print(
    "Columns:",
    feature_dataset.shape[1]
)

print(
    "Missing values:",
    feature_dataset.isnull().sum().sum()
)

print(
    "Duplicate rows:",
    feature_dataset.duplicated().sum()
)

print("\nClass distribution:")

print(
    feature_dataset["label"].value_counts()
)

FINAL FEATURE DATASET VALIDATION
Rows: 1400
Columns: 18
Missing values: 0
Duplicate rows: 0

Class distribution:
label
DangerousDriving    200
Distracted          200
Drinking            200
OpenEye             200
SafeDriving         200
Yawn                200
ClosedEye           200
Name: count, dtype: int64
